# Centered vs. non-centered parameterizations: basic logic

When you fit a hierarchical model in HSSM you implicitly choose between two parameterizations of the group-specific (random) effects. They are statistically equivalent, but **the node names in the underlying PyMC graph differ**, and that difference interacts with the way you specify priors. Misalignment between the two leads to a subtle footgun: a prior you supplied can be silently dropped, leaving a *disconnected* free RV in the graph.

This tutorial walks through:

1. The two parameterizations and the equations behind them.
2. How HSSM/bambi name the PyMC nodes in each case.
3. The disconnected-node footgun, the warning HSSM now emits, and two ways to fix it.

## 1. The two parameterizations

For a group-specific intercept $u_g$ with group-level mean $\mu$ and group-level scale $\sigma$:

**Centered (`noncentered=False`)** — sample the group effect directly:

$$u_g \sim \mathcal{N}(\mu, \sigma)$$

**Non-centered (`noncentered=True`, the bambi default)** — sample a standard-normal offset and rescale:

$$z_g \sim \mathcal{N}(0, 1), \qquad u_g = z_g \cdot \sigma$$

Both produce the same prior distribution on $u_g$ when $\mu = 0$. The non-centered form usually samples better because it decouples $u_g$ from $\sigma$ in the geometry of the posterior. **But notice that the non-centered form does not use $\mu$ at all.** That is the source of the footgun we will see below: a `mu` hyperprior you supply on the group term is silently ignored under non-centered.

## 2. Setup

In [ ]:
import logging
import warnings

import pymc as pm

import hssm

warnings.filterwarnings("ignore")

# Show HSSM warnings inline so we can see the footgun message later.
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
logging.getLogger("hssm").setLevel(logging.WARNING)

cav_data = hssm.load_data("cavanagh_theta")
cav_data.head()

## 3. A centered model

We fit a DDM where the drift rate `v` has a participant-level intercept. We pass `noncentered=False` so bambi builds the term as `1|participant_id ~ Normal(mu, sigma)`.

In [ ]:
model_centered = hssm.HSSM(
    data=cav_data,
    model="ddm",
    include=[
        {
            "name": "v",
            "formula": "v ~ 1 + (1|participant_id)",
            "prior": {
                "Intercept": {"name": "Normal", "mu": 0.0, "sigma": 1.5},
                "1|participant_id": {
                    "name": "Normal",
                    "mu": 0.0,
                    "sigma": {"name": "HalfNormal", "sigma": 0.5},
                },
            },
        }
    ],
    p_outlier=0.0,
    noncentered=False,
)

[rv.name for rv in model_centered.pymc_model.free_RVs]

Under the centered parameterization the group-specific term shows up as a single free RV named `v_1|participant_id`, plus its hyperprior `v_1|participant_id_sigma`. We can confirm this visually with the PyMC graph:

In [ ]:
pm.model_to_graphviz(model_centered.pymc_model)

## 4. A non-centered model

Same prior dictionary, but `noncentered=True` (which is also the default if you do not pass the kwarg).

In [ ]:
model_noncentered = hssm.HSSM(
    data=cav_data,
    model="ddm",
    include=[
        {
            "name": "v",
            "formula": "v ~ 1 + (1|participant_id)",
            "prior": {
                "Intercept": {"name": "Normal", "mu": 0.0, "sigma": 1.5},
                "1|participant_id": {
                    "name": "Normal",
                    "mu": 0.0,
                    "sigma": {"name": "HalfNormal", "sigma": 0.5},
                },
            },
        }
    ],
    p_outlier=0.0,
    noncentered=True,
)

[rv.name for rv in model_noncentered.pymc_model.free_RVs]

Now there is a new free RV called `v_1|participant_id_offset` (the standardized $z_g$), and `v_1|participant_id` is a *deterministic* equal to `offset * sigma`. The naming convention has changed.

In [ ]:
pm.model_to_graphviz(model_noncentered.pymc_model)

## 5. The footgun: a `mu` hyperprior under non-centered

Suppose you write the group prior with a *hyperprior* on the group mean — perfectly natural if you are coming from a centered, fully Bayesian mindset:

```python
"1|participant_id": {
    "name": "Normal",
    "mu": {"name": "Normal", "mu": 0.0, "sigma": 0.5},   # <-- hyperprior on mu
    "sigma": {"name": "HalfNormal", "sigma": 0.5},
}
```

Under `noncentered=True` bambi reparameterizes the term as `offset * sigma` and **never uses `mu`**. The `mu` hyperprior is still created in the PyMC graph (as `v_1|participant_id_mu`), but it is a *disconnected* free RV: it has no path to the observed data. HSSM now warns about this twice — once with a targeted, actionable message, and once with a general "these RVs are not wired in" report.

In [ ]:
model_footgun = hssm.HSSM(
    data=cav_data,
    model="ddm",
    include=[
        {
            "name": "v",
            "formula": "v ~ 1 + (1|participant_id)",
            "prior": {
                "Intercept": {"name": "Normal", "mu": 0.0, "sigma": 1.5},
                "1|participant_id": {
                    "name": "Normal",
                    "mu": {"name": "Normal", "mu": 0.0, "sigma": 0.5},
                    "sigma": {"name": "HalfNormal", "sigma": 0.5},
                },
            },
        }
    ],
    p_outlier=0.0,
    noncentered=True,
)

In [ ]:
from hssm.param.parameterization_check import find_disconnected_free_rvs

find_disconnected_free_rvs(model_footgun.pymc_model)

In [ ]:
pm.model_to_graphviz(model_footgun.pymc_model)

Notice the floating `v_1|participant_id_mu` node — it has no arrow pointing toward the response. That is the orphan.

### Fix 1 — switch to the centered parameterization

If you really want a hyperprior on the group mean, use `noncentered=False` so that `mu` is wired in.

In [ ]:
model_fix_centered = hssm.HSSM(
    data=cav_data,
    model="ddm",
    include=[
        {
            "name": "v",
            "formula": "v ~ 1 + (1|participant_id)",
            "prior": {
                "Intercept": {"name": "Normal", "mu": 0.0, "sigma": 1.5},
                "1|participant_id": {
                    "name": "Normal",
                    "mu": {"name": "Normal", "mu": 0.0, "sigma": 0.5},
                    "sigma": {"name": "HalfNormal", "sigma": 0.5},
                },
            },
        }
    ],
    p_outlier=0.0,
    noncentered=False,
)

find_disconnected_free_rvs(model_fix_centered.pymc_model)

### Fix 2 — move the location prior to the common intercept

If you want to keep the non-centered parameterization (usually a good idea for sampling), express the location of the group mean through the *common* `Intercept` term, which is shared across the model regardless of parameterization.

In [ ]:
model_fix_intercept = hssm.HSSM(
    data=cav_data,
    model="ddm",
    include=[
        {
            "name": "v",
            "formula": "v ~ 1 + (1|participant_id)",
            "prior": {
                # Location goes here, where both parameterizations use it:
                "Intercept": {"name": "Normal", "mu": 0.0, "sigma": 0.5},
                "1|participant_id": {
                    "name": "Normal",
                    "mu": 0.0,  # scalar, not a hyperprior
                    "sigma": {"name": "HalfNormal", "sigma": 0.5},
                },
            },
        }
    ],
    p_outlier=0.0,
    noncentered=True,
)

find_disconnected_free_rvs(model_fix_intercept.pymc_model)

## 6. Rule of thumb

* Default to `noncentered=True` for hierarchical models — it usually samples better, and is also bambi's default.
* Under `noncentered=True`, **the location of a group-specific term lives on the common intercept (or other common terms)**, not on the `mu` argument of the group prior. Set `mu=0` on the group term itself.
* Switch to `noncentered=False` only when you have a specific reason (e.g. you want a hyperprior on the group mean and the geometry of your posterior cooperates).
* When in doubt, plot the PyMC graph with `pm.model_to_graphviz(model.pymc_model)` and look for floating nodes. HSSM also flags them automatically and prints a warning.

## 7. Reference

The warnings come from the `"hssm"` logger. If you want to silence them (not recommended), set its level to `ERROR`:

```python
import logging
logging.getLogger("hssm").setLevel(logging.ERROR)
```

If you want to inspect the graph programmatically, use `hssm.param.parameterization_check.find_disconnected_free_rvs(model.pymc_model)`.